# 06 · Model Karşılaştırma Sonuçları

**Amaç:** `03_train_baseline` ve `05_train_comparison` defterlerinin Drive'a kaydettiği `results.json` ve `history.json` dosyalarını okuyup tek tabloda ve grafiklerde karşılaştırmak.

* Bu defter **eğitim yapmaz** ve **GPU gerektirmez**; veri setini belleğe yüklemez.
* Henüz çalıştırılmamış deney klasörleri otomatik atlanır.
* Üretilen tablo ve şekiller `results/` klasörüne kaydedilir.

In [ ]:
# --- Kurulum: hem Google Colab'da hem kendi bilgisayarında çalışır ---
import json, os, sys
from pathlib import Path

BOOTSTRAP_BRANCH = os.environ.get("TORNADO_BRANCH", "m")

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount("/content/drive")                     # işlenmiş veri ve modeller Drive'da saklanır
    REPO = Path("/content/tornado-cnn")
    cfg_file = REPO / "configs" / "experiments.json"
    branch = json.loads(cfg_file.read_text())["repo"]["branch"] if cfg_file.exists() else BOOTSTRAP_BRANCH
    if not REPO.exists():
        try:                                          # repo private ise: Colab Secrets'taki token
            token = userdata.get("GITHUB_TOKEN") + "@"
        except Exception:                             # repo public ise token gerekmez
            token = ""
        !git clone -q -b {branch} https://{token}github.com/baevv/tornado-cnn.git {REPO}
    else:
        !git -C {REPO} fetch -q origin {branch}
        !git -C {REPO} checkout -q {branch}
        !git -C {REPO} pull -q origin {branch}
    os.chdir(REPO)
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    cfg_file = REPO / "configs" / "experiments.json"
    branch = json.loads(cfg_file.read_text())["repo"]["branch"] if cfg_file.exists() else BOOTSTRAP_BRANCH

sys.path.insert(0, str(REPO))
for _m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_m]

print("Colab" if IN_COLAB else "Yerel", "| repo:", REPO, "| dal:", branch)

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from src.helper import paths

MODELS_ROOT = paths.data_dir().parent / "models"
RESULTS_DIR = REPO / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

exp_registry = paths.load_config("experiments.json")
EXPERIMENTS = {
    item["name"]: item["label"]
    for item in (exp_registry.get("baselines", []) + exp_registry.get("experiments", []))
}

# 05_ ile başlayan ek tam eğitim klasörlerini (ör. çoklu seed) otomatik ekle; _hizli asla alınmaz
if MODELS_ROOT.exists():
    for d in sorted(MODELS_ROOT.iterdir()):
        if d.is_dir() and d.name.startswith("05_") and "_hizli" not in d.name and d.name not in EXPERIMENTS:
            EXPERIMENTS[d.name] = d.name

found, missing = [], []
for folder, label in EXPERIMENTS.items():
    if "_hizli" in folder:
        continue
    r_path = MODELS_ROOT / folder / "results.json"
    if r_path.exists():
        found.append((folder, label))
    else:
        missing.append(folder)

print("Modeller klasörü:", MODELS_ROOT)
print("Bulunanlar:", [f for f, _ in found])
if missing:
    print("Henüz eğitilmemiş (atlandı):", missing)

## 1. Karşılaştırma tablosu

* Dengesiz hortum verisinde doğruluk (`doğruluk`) tek başına yanıltıcı olduğundan tablo ana ölçüt olan **`PR-AUC`** değerine göre azalan sırada listelenir.
* `PR-AUC` ve `ROC-AUC` eşikten bağımsızdır. `F1`, `Recall` ve `Precision` ise doğrulama kümesinde seçilen eşikle hesaplandığından hafifçe iyimser olabilir; nihai tarafsız değerlendirmede bu eşik test kümesinde sabit tutulacaktır.

In [ ]:
# Eski 03_* kayıtlarında `params` alanı yoksa TornadoCNN parametre sayısını doğrudan hesapla
def fallback_params(folder: str):
    if folder.startswith("03_"):
        try:
            from src.helper.model import TornadoCNN, count_params
            return count_params(TornadoCNN())
        except ImportError:
            return 105761
    return None

rows = []
histories = {}
for folder, default_label in found:
    mdir = MODELS_ROOT / folder
    with open(mdir / "results.json", encoding="utf-8") as f:
        res = json.load(f)
    h_path = mdir / "history.json"
    hist = pd.read_json(h_path) if h_path.exists() else pd.DataFrame()
    if not hist.empty:
        histories[folder] = (res.get("model_name", default_label), hist)

    cfg_i = res.get("config", {})
    val = res.get("validation", {})
    best_ep = res.get("best_epoch")
    val_loss = res.get("val_loss")
    if val_loss is None and not hist.empty and best_ep is not None:
        hit = hist.loc[hist["tur"] == best_ep, "doğrulama kaybı"]
        val_loss = float(hit.iloc[0]) if not hit.empty else None

    rows.append({
        "Klasör": folder,
        "Model": res.get("model_name", default_label),
        "Ağırlık": cfg_i.get("weighting", "-"),
        "Parametre": res.get("params") or fallback_params(folder),
        "lr": cfg_i.get("lr"),
        "En İyi Tur": best_ep,
        "Toplam Tur": int(hist["tur"].max()) if not hist.empty else None,
        "Doğrulama Kaybı": val_loss,
        "PR-AUC": val.get("PR-AUC"),
        "ROC-AUC": val.get("ROC-AUC"),
        "F1": val.get("F1"),
        "Recall": val.get("recall (yakalama)"),
        "Precision": val.get("precision (isabet)"),
        "Doğruluk": val.get("doğruluk"),
        "Eşik": res.get("threshold"),
    })

table = pd.DataFrame(rows).sort_values("PR-AUC", ascending=False).reset_index(drop=True)
table.to_csv(RESULTS_DIR / "model_comparison.csv", index=False)

# Flatten teorik parametre notu (eğitilmeden elenme gerekçesi)
print("Not — Flatten kullanılsaydı: 128 × 7 × 15 = 13.440 özellik → Linear(13440, 64) = 860.224 parametre "
      "(toplam 957.729; mevcut Global Max/Avg TornadoCNN ise 105.761 parametre).")
table.round({"Doğrulama Kaybı": 4, "PR-AUC": 4, "ROC-AUC": 4, "F1": 4, "Recall": 4, "Precision": 4, "Doğruluk": 4, "Eşik": 3})

## 2. Öğrenme eğrileri

Üstteki grafiklerde her modelin **eğitim ve doğrulama kaybı** kendi ölçeğinde (farklı `pos_weight` değerleri kayıp ölçeğini değiştirdiği için ayrı eksenlerde), alttaki iki grafikte ise modellerin **doğrulama PR-AUC ve ROC-AUC** gidişatı ortak eksende gösterilir.

In [ ]:
if histories:
    n = len(histories)
    fig, axes = plt.subplots(1, n, figsize=(4.4 * n, 3.6), layout="constrained")
    axes = np.atleast_1d(axes)
    for ax, (folder, (label, hist)) in zip(axes, histories.items()):
        best_ep = int(hist.loc[hist["doğrulama AUC"].idxmax(), "tur"])
        ax.plot(hist["tur"], hist["eğitim kaybı"], "o-", ms=4, label="Eğitim")
        ax.plot(hist["tur"], hist["doğrulama kaybı"], "o-", ms=4, label="Doğrulama")
        ax.axvline(best_ep, color="gray", ls="--", lw=1.2, label=f"En iyi ({best_ep})")
        ax.set_title(label, fontsize=10)
        ax.set_xlabel("Tur (epoch)")
        ax.set_ylabel("Kayıp")
        ax.legend(fontsize=8)
    fig.suptitle("Modellerin Eğitim ve Doğrulama Kayıp Eğrileri")
    fig.savefig(RESULTS_DIR / "learning_curves_loss.png", dpi=150)
    plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.4), layout="constrained")
    for folder, (label, hist) in histories.items():
        axes[0].plot(hist["tur"], hist["doğrulama PR-AUC"], "o-", ms=4, label=label)
        axes[1].plot(hist["tur"], hist["doğrulama AUC"], "o-", ms=4, label=label)
    axes[0].set_title("Doğrulama PR-AUC (Ana Ölçüt)")
    axes[1].set_title("Doğrulama ROC-AUC")
    for ax in axes:
        ax.set_xlabel("Tur (epoch)")
        ax.grid(alpha=0.25)
        ax.legend(fontsize=8)
    fig.savefig(RESULTS_DIR / "learning_curves_auc.png", dpi=150)
    plt.show()

## 3. Doğrulama ölçütlerinin karşılaştırması

In [ ]:
if not table.empty:
    plot_df = table.sort_values("PR-AUC", ascending=True)
    y_pos = np.arange(len(plot_df))
    h = 0.25

    fig, ax = plt.subplots(figsize=(10, max(3.6, 0.9 * len(plot_df))), layout="constrained")
    ax.barh(y_pos + h, plot_df["PR-AUC"], height=h, label="PR-AUC (ana ölçüt)")
    ax.barh(y_pos, plot_df["ROC-AUC"], height=h, label="ROC-AUC")
    ax.barh(y_pos - h, plot_df["F1"], height=h, label="F1")

    for i, (_, r) in enumerate(plot_df.iterrows()):
        ax.text(r["PR-AUC"] + 0.008, y_pos[i] + h, f"{r['PR-AUC']:.3f}", va="center", fontsize=8)
        ax.text(r["ROC-AUC"] + 0.008, y_pos[i], f"{r['ROC-AUC']:.3f}", va="center", fontsize=8)
        ax.text(r["F1"] + 0.008, y_pos[i] - h, f"{r['F1']:.3f}", va="center", fontsize=8)

    ax.set_yticks(y_pos, plot_df["Model"])
    ax.set_xlim(0, 1.02)
    ax.set_xlabel("Doğrulama skoru")
    ax.set_title("Mimari Karşılaştırması (Doğrulama Kümesi)")
    ax.grid(axis="x", alpha=0.25)
    ax.legend(loc="lower right")
    fig.savefig(RESULTS_DIR / "metrics_comparison.png", dpi=150)
    plt.show()